# Januário Leads — Fase 9 V5: 600 mil empresas úteis e distribuídas

**Somente homologação.** O Colab **não inicia sozinho**. Use as quatro células em ordem e confirme `AMPLIAR_600K` apenas quando quiser importar. Não publique chaves e desconecte o runtime ao terminar.

## Estratégia

- IBGE Censo 2022, população por município; grupos: pequeno até 30 mil habitantes (até 100 empresas), médio até 150 mil (até 240), grande até 700 mil (até 700), metrópole acima disso (até 2.400). Municípios novos sem população compatível permanecem com teto conservador 100.
- Dar prioridade a grandes vazios de nichos: alimentação, beleza, saúde, automotivo, academia, hotelaria e comércio, entre outros. Selecionar **apenas estabelecimentos novos, com nome, categoria e contato aproveitável**, verificando ID Overture e nome+localização.
- Preservar 291.515 empresas já importadas. Máximo **+40 novas por município por download**, mas **sem o teto cumulativo +40** da fase V4. Município pode participar de outras sessões até alcançar a sua meta.
- Paradas por **600.000 registros**, **340 MiB na tabela Overture** e **440 MiB no banco inteiro**. Conferir tamanho antes de gravar; manter margem dentro dos 500 MB decimais do Free.
- Downloads com 1 a 3 municípios vizinhos (mesma UF), intervalo >=90 segundos entre inícios; pausa global preventiva de 8 horas após 429 do Overture. Registros de retomada exclusivos da V5 e protegidos no Supabase.
- Geoapify e Google Places **não são usados na importação** e permanecem disponíveis como complementos na busca do SaaS.

**Meta de 600 mil é uma aspiração, não garantia:** a fonte pode não conter empresas suficientes com contatos válidos, e o limite de armazenamento pode ser atingido antes. As categorias e municípios elegíveis são escolhidos com dados atuais do próprio Supabase e população oficial do IBGE.

Os scripts do Colab são obtidos do repositório `wesleyshoto702-hub/januarioleads` e a chave de homologação fica somente em memória do Colab. Os checkpoints da V4 não são modificados.

In [ ]:
# 1 — Instalar dependências necessárias somente quando executar
%pip install -q "overturemaps>=2.1,<3" "geopandas>=1,<2" "shapely>=2,<3" "pyogrio>=0.10,<1"
print("Dependências instaladas. Execute a próxima célula.")

In [ ]:
# 2 — Planejamento: baixar scripts do repositório público e população IBGE (SEM chaves)
import collections, datetime as dt, getpass, json, math, os, random, re, subprocess, sys, time, urllib.parse, urllib.request
from pathlib import Path
BASE="https://raw.githubusercontent.com/wesleyshoto702-hub/januarioleads/main/scripts/"
STAGING_URL="https://prlrxjothrslhswpkoza.supabase.co"
WORK=Path("/content"); WORK.mkdir(exist_ok=True)
for filename in ("expand_overture_to_100_staging.py","expand_overture_600k_staging.py"):
    url=BASE+filename
    req=urllib.request.Request(url,headers={"User-Agent":"JanuarioLeads-Import/1.0"})
    with urllib.request.urlopen(req,timeout=40) as response:
        source=response.read(120000)
    if len(source)>90000 or not source.startswith(b"#!/usr/bin/env python3"):
        raise RuntimeError("Código do importador inválido; cancelado sem consultar o Overture.")
    (WORK/filename).write_bytes(source)
sys.path.insert(0,str(WORK))
import expand_overture_to_100_staging as v4
import expand_overture_600k_staging as v5
assert v5.STAGING_URL==STAGING_URL and v5.GOAL_COMPANIES==600000
assert v5.OVERTURE_MAX_MIB==340 and v5.DATABASE_MAX_MIB==440

# IBGE Censo 2022: população municipal (Agregados 4714, variável 93)
IBGE_UF={"11":"RO","12":"AC","13":"AM","14":"RR","15":"PA","16":"AP","17":"TO",
"21":"MA","22":"PI","23":"CE","24":"RN","25":"PB","26":"PE","27":"AL",
"28":"SE","29":"BA","31":"MG","32":"ES","33":"RJ","35":"SP",
"41":"PR","42":"SC","43":"RS","50":"MS","51":"MT","52":"GO","53":"DF"}
def read_public_json(url):
    if not url.startswith("https://servicodados.ibge.gov.br/api/"):
        raise ValueError("Apenas dados públicos oficiais do IBGE")
    req=urllib.request.Request(url,headers={"User-Agent":"JanuarioLeads-Population-Study/1.0"})
    with urllib.request.urlopen(req,timeout=55) as response:
        payload=response.read(14000000)
    return json.loads(payload.decode("utf-8"))

def population_index():
    municip=read_public_json("https://servicodados.ibge.gov.br/api/v1/localidades/municipios")
    names={}
    for r in municip:
        uf=(r.get("microrregiao") or {}).get("mesorregiao",{}).get("UF",{}).get("sigla")
        if not uf:
            uf=IBGE_UF.get(str(r["id"])[:2])
        if uf:
            names[str(r["id"])]=(uf,v4.normalize(r["nome"]))
    agreg=read_public_json("https://servicodados.ibge.gov.br/api/v3/agregados/4714/periodos/2022/variaveis/93?localidades=N6%5Ball%5D")
    result={}
    for block in agreg:
        if str(block.get("id"))!="93":continue
        for group in block.get("resultados",[]):
            for series in group.get("series",[]):
                loc=series.get("localidade") or {}
                code=str(loc.get("id") or "")
                value=(series.get("serie") or {}).get("2022")
                key=names.get(code)
                if key and str(value or "").isdigit():
                    result[key]=int(value)
    if len(result)<5400:
        raise RuntimeError("População oficial incompleta: nenhuma importação será iniciada.")
    return result

POPULATIONS=population_index()
print("IBGE: população encontrada para",len(POPULATIONS),"municípios; origem: Censo 2022")

def plan_for_city(row):
    population=POPULATIONS.get((row["state"],row["city_key"]))
    if population is None:tier="unknown"
    elif population<30000:tier="small"
    elif population<150000:tier="medium"
    elif population<700000:tier="large"
    else:tier="metro"
    return {"tier":tier,"target":v5.CITY_CAPS[tier],"population":population}

def read_stage(secret,path):
    return v4.api(STAGING_URL,secret,path)

def paged_stage(secret,table,fields):
    output=[]; offset=0
    while True:
        qs=urllib.parse.urlencode({"select":fields,"order":"state.asc,city_key.asc",
                                   "limit":"1000","offset":str(offset)})
        page=read_stage(secret,"/rest/v1/"+table+"?"+qs)
        if not isinstance(page,list):
            raise RuntimeError("Falha na leitura dos dados de homologação")
        output.extend(page)
        if len(page)<1000:break
        offset+=len(page)
    return output

def priority(row):
    level={"metro":4,"large":3,"medium":2,"small":1,"unknown":0}[row["plan"]["tier"]]
    # Prioritize high-population cities with few niche groups represented.
    niches=sum(int(row.get(k,0) or 0)==0 for k in (
        "barbers","food","fitness","beauty","health","auto_services","lodging","retail"))
    return (level,niches,-int(row["imported_rows"]),row["plan"]["target"]-int(row["imported_rows"]))

def ordered_queue(rows,blocked):
    by_state=collections.defaultdict(list)
    for row in rows:
        key=(row["state"],row["city_key"])
        if key in blocked:continue
        row["plan"]=plan_for_city(row)
        if int(row["imported_rows"])>=row["plan"]["target"]:continue
        by_state[row["state"]].append(row)
    for uf in by_state:
        by_state[uf].sort(key=priority,reverse=True)
    ordered=[]
    while any(by_state.values()):
        for state in sorted(by_state):
            if by_state[state]:ordered.append(by_state[state].pop(0))
    return ordered

def close_groups(rows):
    groups=[];unused={(r["state"],r["city_key"]) for r in rows}
    by_state=collections.defaultdict(list)
    for row in rows:by_state[row["state"]].append(row)
    for row in rows:
        key=(row["state"],row["city_key"])
        if key not in unused:continue
        unused.remove(key);group=[row]
        for other in by_state[row["state"]]:
            k=(other["state"],other["city_key"])
            if k not in unused:continue
            if all(v4.normalize(str(other["city"]))!=v4.normalize(str(x["city"])) and
                   distance_km(other,x)<=25 for x in group):
                group.append(other);unused.remove(k)
                if len(group)>=3:break
        groups.append(group)
    return groups

def distance_km(a,b):
    try:
        x1,y1=float(a["center_lat"]),float(a["center_lon"])
        x2,y2=float(b["center_lat"]),float(b["center_lon"])
        h=math.sin(math.radians(x2-x1)/2)**2+math.cos(math.radians(x1))*math.cos(math.radians(x2))*math.sin(math.radians(y2-y1)/2)**2
        return 12742*math.asin(min(1,math.sqrt(h)))
    except (ValueError,TypeError,KeyError):return float("inf")

def recent_rate_limit(secret):
    times=[]
    for table in ("overture_phase9_attempts","overture_phase9_v5_attempts"):
        q=urllib.parse.urlencode({"select":"attempted_at","error_code":"eq.fonte_taxa",
                                  "order":"attempted_at.desc","limit":"1"})
        res=read_stage(secret,"/rest/v1/"+table+"?"+q)
        if res:times.append(dt.datetime.fromisoformat(res[0]["attempted_at"].replace("Z","+00:00")))
    return max(times) if times else None

MIN_INTERVAL=90
COOLDOWN_HOURS=8
MAX_DOWNLOADS_PER_SESSION=120
MAX_RUN_SECONDS=9*3600
assert MIN_INTERVAL>=75 and COOLDOWN_HOURS>=6
assert v5.MAX_NEW_PER_BATCH==40 and v5.MAX_CITIES_PER_BATCH==3
print("TESTE LOCAL OK: cotas escalonadas, 600 mil, pausa, agrupamento e proteção de banco.")


In [ ]:
# 3 — EXECUÇÃO EXPLÍCITA: confirmação e credencial oculta da HOMOLOGAÇÃO
def ampliar_600k():
    print("FASE 9 V5 — até 600.000 empresas úteis; Overture <=340 MiB; banco <=440 MiB.")
    print("Máx. 40 novos por cidade/download; até 3 cidades vizinhas; 90s entre downloads.")
    confirm=input("Para importar SOMENTE na HOMOLOGAÇÃO, digite AMPLIAR_600K: ").strip()
    if confirm!="AMPLIAR_600K":
        print("CANCELADO. Não houve importação.")
        return
    secret=getpass.getpass("Chave SECRETA da HOMOLOGAÇÃO (oculta): ").strip()
    if not (secret.startswith("sb_secret_") or secret.startswith("eyJ")):
        print("Chave de homologação inválida; abortado.")
        return
    try:
        last=recent_rate_limit(secret)
        now=dt.datetime.now(dt.timezone.utc)
        if last and now<last+dt.timedelta(hours=COOLDOWN_HOURS):
            print("PARADA SEGURA: respeitar o Overture até",
                  (last+dt.timedelta(hours=COOLDOWN_HOURS)).isoformat())
            return
        stats=v5.current_stats(STAGING_URL,secret)
        print("Empresas no banco:",stats["total_companies"],
              "| Overture MiB:",round(stats["overture_bytes"]/v5.MIB,2),
              "| Banco MiB:",round(stats["database_bytes"]/v5.MIB,2))
        if v5.safe_capacity(stats,1)==0:
            print("PARADA: meta nacional ou espaço insuficiente")
            return
        gaps=paged_stage(secret,"overture_phase9_v5_gaps",
          "state,city,city_key,imported_rows,barbers,food,fitness,beauty,health,auto_services,lodging,retail,with_phone,with_email,with_website")
        centers=paged_stage(secret,"overture_phase9_city_centers",
          "state,city_key,center_lat,center_lon")
        loc={(x["state"],x["city_key"]):x for x in centers}
        for row in gaps:
            center=loc.get((row["state"],row["city_key"]),{})
            row.update(center_lat=center.get("center_lat"),center_lon=center.get("center_lon"))
        checkpoints=paged_stage(secret,"overture_phase9_v5_attempts",
          "state,city_key,result,attempted_at,error_code")
        # The V5 checkpoint is NOT a permanent per-city +40 cap.
        # Only avoid repeating the same city inside one execution.
        targets=ordered_queue(gaps,set())
        tier_counts=collections.Counter(x["plan"]["tier"] for x in targets)
        theoretical=sum(min(x["plan"]["target"],max(x["imported_rows"],0))
                        for x in gaps if "plan" in x)
        print("Municípios ainda abaixo das metas:",len(targets),
              "| por porte:",dict(tier_counts))
        if not targets:
            print("Sem cidades pendentes para esta configuração.")
            return
        seen=set()
        attempts=success=added=errors=0
        next_download_at=time.monotonic()
        deadline=time.monotonic()+MAX_RUN_SECONDS
        groups=close_groups(targets)
        for group in groups:
            if attempts>=MAX_DOWNLOADS_PER_SESSION or time.monotonic()>deadline-240:
                print("PARADA SEGURA: limite da sessão atingido")
                break
            group=[r for r in group if (r["state"],r["city_key"]) not in seen]
            if not group:continue
            now=dt.datetime.now(dt.timezone.utc)
            stats=v5.current_stats(STAGING_URL,secret)
            if v5.safe_capacity(stats,1)==0:
                print("PARADA: teto nacional/armazenamento atingido.")
                break
            delay=max(0,next_download_at-time.monotonic())
            if time.monotonic()+delay>deadline-240:
                print("PARADA: janela do Colab perto do final.")
                break
            if delay:time.sleep(delay)
            next_download_at=time.monotonic()+MIN_INTERVAL+random.uniform(0,15)
            state=group[0]["state"]
            names=[r["city"] for r in group]
            plans={r["city_key"]:r["plan"] for r in group}
            command=[sys.executable,str(WORK/"expand_overture_600k_staging.py"),
                     "--states",state,"--cities",",".join(names),
                     "--plans-json",json.dumps(plans,ensure_ascii=False)]
            env=os.environ.copy()
            env["SUPABASE_URL"]=STAGING_URL
            env["SUPABASE_SERVICE_ROLE_KEY"]=secret
            attempts+=1
            print("LOTE",attempts,"UF",state,"| municípios:",", ".join(names),flush=True)
            try:
                result=subprocess.run(command,env=env,text=True,capture_output=True,
                                      timeout=min(1500,int(max(120,deadline-time.monotonic()-120))))
                output=(result.stdout or "")+"\n"+(result.stderr or "")
                code=None if result.returncode==0 else v4_error(output,result.returncode)
            except subprocess.TimeoutExpired:
                output="";code="timeout_subprocesso"
            # Never echo subprocess stderr (might contain sensitive request headers).
            if code:
                errors+=1
                print("ERRO SANITIZADO:",code)
                for row in group:
                    key=row["city_key"]
                    try:
                        previous=len(v5.existing_city(STAGING_URL,secret,state,key,row["plan"]["target"]))
                        v5.record_checkpoint(STAGING_URL,secret,state,key,row["city"],
                                             row["plan"],previous,previous,"error",code)
                    except Exception:
                        print("PARADA: checkpoint indisponível.")
                        return
                if code in ("fonte_taxa","fonte_acesso","supabase_auth",
                            "supabase_taxa","limite_banco","dependencia_ausente"):
                    print("PARADA GLOBAL:",code,"; não repetir a consulta à fonte agora.")
                    break
                if errors>=3:
                    print("PARADA: 3 falhas; verifique o ambiente sem repetir a fonte.")
                    break
            else:
                success+=1
                for line in output.splitlines():
                    if line.startswith("CITY_RESULT ") or line.startswith("BATCH_SUMMARY "):
                        print(line[:1000])
                        if line.startswith("CITY_RESULT "):
                            try:added+=int(json.loads(line[len("CITY_RESULT "):])["added"])
                            except (ValueError,KeyError,TypeError):pass
            for row in group:
                seen.add((row["state"],row["city_key"]))
        print("RESUMO V5: downloads",attempts,"| lotes OK",success,
              "| novas empresas contabilizadas",added,"| erros",errors)
        stats=v5.current_stats(STAGING_URL,secret)
        print("FINAL: empresas",stats["total_companies"],
              "| Overture MiB",round(stats["overture_bytes"]/v5.MIB,2),
              "| banco MiB",round(stats["database_bytes"]/v5.MIB,2))
    except Exception as exc:
        print("PARADA SEGURA: ocorreu uma falha de configuração ou serviço:",type(exc).__name__)
    finally:
        secret=""
        print("Chave descartada em memória local. Desconecte o Colab ao terminar.")

def v4_error(text,code):
    return v4.classify_import_error(text,code)

ampliar_600k()